# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, referencing all key entities by their `@id`.

### Dataset Source
The dataset is described via the Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata (not subscripted, use attributes directly)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n\nPublished: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
List available record sets and their fields by `@id`. You will want to inspect which record sets exist, and for each one, what field IDs are available.

In [ ]:
# Explore the record sets and fields by @id
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for record_set in record_sets:
    print(f"RecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name if hasattr(record_set, 'name') else ''}")
    print(f"  Description: {record_set.description if hasattr(record_set, 'description') else ''}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    Field @id: {field.id}\tName: {field.name if hasattr(field, 'name') else ''}\tDataType: {getattr(field, 'data_type', '')}")
    print()

## 3. Data Extraction
Load data from each record set into pandas DataFrames, referencing the record set and field `@id`s discovered above.

In [ ]:
# Load all records into DataFrames, indexed by RecordSet @id
dataframes = {}
for rs in dataset.record_sets:
    records = list(dataset.records(record_set=rs.id))  # Always reference record set by @id
    if records:
        dataframes[rs.id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from RecordSet @id: {rs.id}")
        print(f"Fields: {dataframes[rs.id].columns.tolist()}")
        print()

if not dataframes:
    print("No dataframes loaded; check schema or data availability.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (for example, the protein 'coverage' or 'peptides_count') from the main record set and perform basic analysis: filtering, normalization, and grouping. Refer to the fields by their `@id`.

In [ ]:
# Select a record set with quantitative data (update as appropriate)
# For demonstration: lets use the first DataFrame if available
if dataframes:
    # Pick the first record set
    record_set_id = list(dataframes.keys())[0]  # Use @id
    df = dataframes[record_set_id]
    print(f"Preview of data in RecordSet @id {record_set_id}:")
    display(df.head())

    # Try to find a numeric field to analyze (e.g., 'coverage', 'peptides_count', might vary by this dataset)
    # If known, replace with correct field @id or column name
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_candidates:
        print("No numeric columns found for EDA.")
    else:
        numeric_field = numeric_candidates[0]  # Pick the first numeric column
        group_field = None
        # Try to find a non-numeric field for grouping
        for col in df.columns:
            if df[col].dtype == object and col != numeric_field:
                group_field = col
                break
        # Filtering
        threshold = df[numeric_field].mean() if len(df) > 0 else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std(ddof=0)
        print(f"Normalized {numeric_field}:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group by
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
else:
    print("No DataFrames loaded for EDA.")

## 5. Visualization
Visualize the distribution of one of the main numeric fields. For group analysis, visualize group averages if available.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = dataframes[record_set_id]
    if numeric_candidates:
        plt.figure(figsize=(8,5))
        sns.histplot(df[numeric_field], kde=True, bins=30)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()

        if 'grouped_df' in locals():
            plt.figure(figsize=(10,4))
            sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
            plt.title(f"Mean {numeric_field} by {group_field}")
            plt.xticks(rotation=45)
            plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR² Mass Spectrometry dataset using the `mlcroissant` library, explored its record sets and fields by `@id`, performed exploratory analysis on numeric variables, normalized and grouped the data, and visualized distributions. This workflow demonstrates a FAIR and reproducible approach to scientific data exploration.
